## New notebook to read output of MD simulation, using Adios2 library

### part 1: **print information summary**

- Reading one of the sample output files (.bp), step by step.  
- Printing available variables and attributes, as well as their structure.

In [ ]:
import numpy as np
from adios2 import Stream
import os

def read_adios_output(outputDir, index, print_summary=True):
    attributes = {}
    variables = {}
    
    filename = os.path.join(outputDir, f"run_{index}.bp")

    with Stream(filename, "r") as s:
        for i, _ in enumerate(s.steps()):
            if i == 0:
                for attr in s.available_attributes():
                    attributes[attr] = s.read_attribute(attr)
            for var in s.available_variables():
                if var not in variables:
                    variables[var] = []
                variables[var].append(s.read(var))

    for var in variables:
        variables[var] = np.array(variables[var])

    if print_summary:
        print("Attributes Summary:")
        print("-------------------")
        for idx, (name, value) in enumerate(attributes.items(), start=1):
            print(f"{idx}. {name:<40} {value}")
        print("\nVariables Summary:")
        print("------------------")
        for idx, (name, arr) in enumerate(variables.items(), start=1):
            shape = arr.shape
            if arr.ndim == 1:
                description = shape[0]
            else:
                description = f"{shape[0]} * {list(shape[1:])}"
            print(f"{idx}. {name:<35} {description}")

    return {"attributes": attributes, "variables": variables}

outputDir = "/home/hadis/custom_vector/buildParticleOriented/buildVS/feb1_testAdios/outputs/"
result = read_adios_output(outputDir, 0)


---

### Part 2: **Read and Plot Neighbor Counts**

In [42]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from adios2 import FileReader
from scipy.signal import savgol_filter

def read_variable(output_dir, variable_name):
    """ gets the variable data, temperature, and run index from the output directory """
    
    pattern = re.compile("run_([0-9]+).bp")
    run_numbers = [int(pattern.match(x)[1]) for x in os.listdir(output_dir) if pattern.match(x)]
    
    if not run_numbers:
        raise ValueError(f"No run files found in {output_dir}")
    
    run_numbers.sort()
    variable_data = []
    temperatures = []
    run_indeces = []
    
    for run_num in run_numbers:
        file_path = os.path.join(output_dir, f"run_{run_num}.bp")
        with FileReader(file_path) as reader:
            if variable_name in reader.available_variables():
                var_info = reader.available_variables()[variable_name]
                steps = int(var_info.get("AvailableStepsCount", 1))
                
                data = reader.read(variable_name, step_selection=[steps - 1, 1])
                data = data.flatten()
                variable_data.append(data)
                
                temp_label = reader.read_attribute("temperature")
                temp_label = temp_label.flatten()
                temperatures.append(temp_label)
                
                run_index = reader.read_attribute("runIndex")
                run_index = run_index.flatten()
                run_indeces.append(run_index)
            else:
                raise ValueError(f"Variable {variable_name} not found in {file_path}")
            
    return np.array(variable_data), np.array(temperatures), np.array(run_indeces)

def plot_neighbors(neighbors_array, m_temperatures, smoothing=True, window_length=10, polyorder=3):
    neighbors_data = neighbors_array[0]
    temp_labels = neighbors_array[1]
    run_indeces = neighbors_array[2]
    
    temperature_data = m_temperatures[0]
    
    fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(7, 8))
    
    for ax in axs:
        ax.grid(True, linestyle='--', alpha=0.7, color='gray')
    
    colors = plt.cm.viridis(np.linspace(0, 1, neighbors_data.shape[1]))
    
    for i in range(neighbors_data.shape[1]):
        y = neighbors_data[:, i]
        if smoothing:
            y = savgol_filter(y, window_length, polyorder)
        axs[0].plot(y, label=f"Shell {i}", color=colors[i], linewidth=2)
    
    total_points = len(run_indeces)
    tick_indices = np.linspace(0, total_points-1, 10, dtype=int)
    
    axs[0].set_xticks(tick_indices)
    axs[0].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in tick_indices], rotation=45)
    axs[0].set_ylabel("Number of Neighbors")
    axs[0].set_title("Neighbor Analysis")
    
    axis_labels = ['x', 'y', r'$\omega$']
    
    for i, axis_label in enumerate(axis_labels):
        y = temperature_data[:, i]
        if smoothing:
            y = savgol_filter(y, window_length, polyorder)
        axs[1].plot(y, label=f"T{axis_label}", color=colors[i], linewidth=2)
    
    axs[1].set_xlabel("Temperature Labels")
    axs[1].set_xticks(tick_indices)
    axs[1].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in tick_indices], rotation=45)
    # axs[1].set_xticklabels([f"run {run_indeces[i][0]:.0f}" for i in tick_indices], rotation=45)
    axs[1].set_ylabel("Measured Temperature")

    for ax in axs:
        ax.legend(bbox_to_anchor=(1, 1), loc='upper left', frameon=True, fancybox=True, shadow=False)
    
    plt.tight_layout()
    plt.show()

def plot_energies(kinetic_energy, potential_energy, m_temperatures, show_potential=True, smoothing=True, window_length=10, polyorder=3):
    kinetic_energy_data = kinetic_energy[0]
    potential_energy_data = potential_energy[0]/ 100
    temp_labels = kinetic_energy[1]
    run_indeces = kinetic_energy[2]
    
    temperature_data = m_temperatures[0]
    
    fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(7, 8))
    
    for ax in axs:
        ax.grid(True, linestyle='--', alpha=0.7, color='gray')
    
    colors = plt.cm.plasma(np.linspace(0, 1, kinetic_energy_data.shape[1]+3))
    
    axis_labels = ['x', 'y', r'$\omega$']
    
    for i, axis_label in enumerate(axis_labels):
        y = kinetic_energy_data[:, i]
        if smoothing:
            y = savgol_filter(y, window_length, polyorder)
        axs[0].plot(y, label=f"K{axis_label}", color=colors[i])
    if show_potential:    
        y2 = potential_energy_data.flatten()
        if smoothing:
            y2 = savgol_filter(y2, window_length, polyorder)
        axs[0].plot(y2, label='U', color=colors[3])
        axs[0].plot(y2 + np.sum(kinetic_energy_data, axis=1), label='Total', color=colors[4])
    
    total_points = len(run_indeces)
    tick_indices = np.linspace(0, total_points-1, 10, dtype=int)
    
    axs[0].set_xticks(tick_indices)
    axs[0].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in tick_indices], rotation=45)
    axs[0].set_ylabel("Energy")
    axs[0].set_title("Energy Evolution")
     
    for i, axis_label in enumerate(axis_labels):
        y = temperature_data[:, i]
        if smoothing:
            y = savgol_filter(y, window_length, polyorder)
        axs[1].plot(y, label=f"T{axis_label}", color=colors[i])
    
    axs[1].set_xlabel("Temperature Labels")
    axs[1].set_xticks(tick_indices)
    axs[1].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in tick_indices], rotation=45)
    # axs[1].set_xticklabels([f"run {run_indeces[i][0]:.0f}" for i in tick_indices], rotation=45)
    axs[1].set_ylabel("Measured Temperature")

    for ax in axs:
        ax.legend(bbox_to_anchor=(1, 1), loc='upper left', frameon=True, fancybox=True, shadow=False)
    
    plt.tight_layout()
    
    plt.show()

def plot_com_velocity(com_velocity, m_temperatures, smoothing=True, window_length=10, polyorder=3):
    com_vel_data = com_velocity[0]
    temp_labels = com_velocity[1]
    run_indeces = com_velocity[2]
    
    temperature_data = m_temperatures[0]
    
    fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(7, 8))
    
    for ax in axs:
        ax.grid(True, linestyle='--', alpha=0.7, color='gray')
    
    colors = plt.cm.plasma(np.linspace(0, 1, 4))
    
    axis_labels = ['x', 'y']
    # axis_labels = ['x', 'y', r'$\omega$']  #later edit COM calculation
    
    for i, axis_label in enumerate(axis_labels):
        y = com_vel_data[:, i]
        if smoothing:
            y = savgol_filter(y, window_length, polyorder)
        axs[0].plot(y, label=f"K{axis_label}", color=colors[i])
    
    total_points = len(run_indeces)
    tick_indices = np.linspace(0, total_points-1, 10, dtype=int)
    
    axs[0].set_xticks(tick_indices)
    axs[0].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in tick_indices], rotation=45)
    axs[0].set_ylabel("Velocity")
    axs[0].set_title("Center of Mass Velocity Evolution")
     
    for i, axis_label in enumerate(axis_labels):
        y = temperature_data[:, i]
        if smoothing:
            y = savgol_filter(y, window_length, polyorder)
        axs[1].plot(y, label=f"T{axis_label}", color=colors[i])
    
    axs[1].set_xlabel("Temperature Labels")
    axs[1].set_xticks(tick_indices)
    axs[1].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in tick_indices], rotation=45)
    # axs[1].set_xticklabels([f"run {run_indeces[i][0]:.0f}" for i in tick_indices], rotation=45)
    axs[1].set_ylabel("Measured Temperature")

    for ax in axs:
        ax.legend(bbox_to_anchor=(1, 1), loc='upper left', frameon=True, fancybox=True, shadow=False)
    
    plt.tight_layout()
    
    plt.show()

def plot_ave_velocities(velocities, m_temperatures, smoothing=True, window_length=10, polyorder=3, particle_oriented=False):
    velocities_data = velocities[0]
    temp_labels = velocities[1]
    run_indeces = velocities[2]
    
    temperature_data = m_temperatures[0]

    num_particles = 49
    num_components = 3 if particle_oriented else 2

    reshaped_velocities = velocities_data.reshape(-1, num_particles, num_components)

    avg_velocities = np.mean(reshaped_velocities, axis=1)

    fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(7, 8))

    for ax in axs:
        ax.grid(True, linestyle='--', alpha=0.7, color='gray')

    axis_labels = ['Vx', 'Vy']
    if particle_oriented:
        axis_labels.append(r'$V_{\phi}$')

    colors = plt.cm.plasma(np.linspace(0, 1, 4))

    # Plotting the average velocities
    for i, axis_label in enumerate(axis_labels):
        y = avg_velocities[:, i]
        
        if smoothing:
            y = savgol_filter(y, window_length, polyorder)
        axs[0].plot(y, label=axis_label, color=colors[i])

    total_points = len(run_indeces)
    tick_indices = np.linspace(0, total_points - 1, 10, dtype=int)

    axs[0].set_xticks(tick_indices)
    axs[0].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in tick_indices], rotation=45)
    axs[0].set_ylabel("Average Velocity")
    axs[0].set_title("Average Velocity Evolution")

    # Plotting measured temperatures
    temp_labels_for_axes = ['Tx', 'Ty', r'$T_{\omega}$']
    for i, temp_label in enumerate(temp_labels_for_axes):
        y = temperature_data[:, i]
        if smoothing:
            y = savgol_filter(y, window_length, polyorder)
        axs[1].plot(y, label=temp_label, color=colors[i])

    axs[1].set_xlabel("Temperature Labels")
    axs[1].set_xticks(tick_indices)
    axs[1].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in tick_indices], rotation=45)
    axs[1].set_ylabel("Measured Temperature")

    for ax in axs:
        ax.legend(bbox_to_anchor=(1, 1), loc='upper left', frameon=True, fancybox=True, shadow=False)

    plt.tight_layout()
    plt.show()


In [38]:
output_dir = "/home/hadis/custom_vector/buildParticleOriented/buildVS/feb1_testAdios/outputs/"
temperature = read_variable(output_dir, "real temperature")

In [ ]:
neighbors = read_variable(output_dir, "number of neighbors")
plot_neighbors(neighbors, temperature)

In [36]:
kinetic_energy = read_variable(output_dir, "kinetic energy")
potential_energy = read_variable(output_dir, "potential energy")

plot_energies(kinetic_energy, potential_energy, temperature, show_potential=False)

In [ ]:
com_velocity = read_variable(output_dir, "center of mass velocity")
plot_com_velocity(com_velocity, temperature)

In [ ]:
velocity = read_variable(output_dir, "velocities")
plot_ave_velocities(velocity, temperature)